# Schema Validation

1. What is Schema-on-read / Schema-on-write? 
2. Column order validation
3. Data Type validation
4. Column name validation
5. Nullability validation
6. Extra columns validation

In [0]:
DROP TABLE deltacatalog.deltadb.invoices_sv;
CREATE OR REPLACE TABLE deltacatalog.deltadb.invoices_sv (
  customer_id INT NOT NULL,  
  invoice_no STRING,
  quantity INT,
  price FLOAT, 
  invoice_date DATE
); 

INSERT INTO deltacatalog.deltadb.invoices_sv
SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_1_100.parquet`; 

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_sv
LIMIT 5;

In [0]:
SELECT MIN(customer_id), MAX(customer_id), COUNT(*)
FROM deltacatalog.deltadb.invoices_sv;

#### Scenario 1: Column Order Validation

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_sv
SELECT quantity, invoice_no, customer_id, price, invoice_date
FROM VALUES (99999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_sv
WHERE customer_id = 99999;

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_sv
WHERE customer_id = 10;

In [0]:
SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_101_200.parquet`
ORDER BY customer_id DESC LIMIT 5;

In [0]:
MERGE INTO deltacatalog.deltadb.invoices_sv AS tgt
USING (
  SELECT quantity, invoice_no, customer_id, CAST (price AS FLOAT), invoice_date
  FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY customer_id DESC LIMIT 5
) src 
ON tgt.customer_id = src.customer_id
WHEN MATCHED THEN 
  UPDATE SET 
    tgt.customer_id = src.customer_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.invoice_date = current_date
WHEN NOT MATCHED THEN
  INSERT * 

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_sv
WHERE customer_id > 100; 

#### Scenario 2: Data Type Validation

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_sv
VALUES (
  'ABC', 
  'I45678',
  10,
  98.75,
  '2025-01-01'
)

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_sv
VALUES (
  '99499', 
  'I45678',
  10,
  98.75,
  '2025-01-01'
)

#### Scenario 3: Column Name Validation

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_sv
SELECT customer_id AS c_id, invoice_no, quantity AS qty, price, invoice_date
FROM VALUES (99999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_sv
WHERE customer_id = 99999;

In [0]:
SELECT customer_id, invoice_no, quantity, CAST (price AS FLOAT), invoice_date
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_101_200.parquet`
ORDER BY customer_id LIMIT 5

In [0]:
MERGE INTO deltacatalog.deltadb.invoices_sv AS tgt
USING (
  SELECT customer_id AS c_id, invoice_no, quantity AS qty, CAST (price AS FLOAT), invoice_date
  FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  ORDER BY customer_id LIMIT 5
) src 
ON tgt.customer_id = src.c_id
WHEN MATCHED THEN 
  UPDATE SET 
    tgt.customer_id = src.c_id,
    tgt.invoice_no = src.invoice_no,
    tgt.quantity = src.qty,
    tgt.price = src.price,
    tgt.invoice_date = current_date
WHEN NOT MATCHED THEN
  INSERT * 

In [0]:
SELECT * FROM deltacatalog.deltadb.invoices_sv
WHERE customer_id > 100;

#### Scenario 4: Nullability Validation

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_sv
VALUES (NULL, NULL, NULL, NULL, NULL);

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_sv
VALUES (78912, NULL, NULL, NULL, NULL);

#### Scenario 5: Extra Column Validation

In [0]:
INSERT INTO deltacatalog.deltadb.invoices_sv
SELECT customer_id, invoice_no, quantity, price, invoice_date, "VIP" AS customer_type
FROM VALUES (78999, 'I12345', 10, 100, '2025-01-01')
AS T(customer_id, invoice_no, quantity, price, invoice_date)

In [0]:
SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_101_200.parquet`
WHERE customer_id BETWEEN 150 AND 155

In [0]:
MERGE INTO deltacatalog.deltadb.invoices_sv AS tgt
USING (
  SELECT customer_id, invoice_no, quantity, price, invoice_date, "VIP" AS customer_type
  FROM PARQUET.`abfss://labdata@dbdeltalabstorageacct.dfs.core.windows.net/invoices/invoices_101_200.parquet`
  WHERE customer_id BETWEEN 150 AND 155
) src 
ON tgt.customer_id = src.customer_id
WHEN NOT MATCHED THEN
  INSERT * 

In [0]:
SELECT customer_id, invoice_no, quantity, price, invoice_date
FROM deltacatalog.deltadb.invoices_sv
WHERE customer_id BETWEEN 150 AND 155